In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()



In [2]:

from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)


In [4]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [5]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)



To run Ollama locally:

1. Install Ollama from https://ollama.com/download for your operating system.
   - macOS: download the `.pkg` and install it.
   - Windows: download the `.msi` and install it.
   - Linux: run:
   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. Open a terminal and start a model locally:
```bash
ollama run llama3
```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

3. To test that the local server is running, use:
```bash
curl http://localhost:11434
```

You should get a response like:
```json
{"models": [...]}  
```

If you want to use Ollama from Python, install the client with:
```bash
pip install ollama
```


In [6]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

I don’t see any context about **“Olama”** specifically.

If you mean running the **course locally**, the FAQ says you can do that instead of using Codespaces, as long as you’re comfortable setting up:

- Python
- `uv`
- Jupyter
- Docker
- any other tools needed for the module

It also says to **document your setup** and keep the environment **reproducible**.

If you meant a different tool name, please clarify and I can answer based on the FAQ context.


In [7]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Yes—usually you can join, but it depends on the course’s enrollment rules and whether it’s still open.\n\nA few quick things to check:\n- **Enrollment deadline**: Is it still open?\n- **Prerequisites**: Do you meet any required background or prior courses?\n- **Capacity**: Are there any seats left?\n- **Approval needed**: Some courses require instructor or department permission.\n\nIf you want, I can help you draft a short message to the instructor or registrar asking to join.'

In [8]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [10]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [11]:
len(response.output)
call = response.output[0]
call



ResponseFunctionToolCall(arguments='{"query":"Can I join after the course has started? discovered course late join"}', call_id='call_MDZPWyG8Inh2DZy2vd6U4xVF', name='search', type='function_call', id='fc_0bbea0dc257b9535006a35d9fb7138819ba4028985528b9324', namespace=None, status='completed')

In [12]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [14]:
result_json

'[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "answer": "No, you can only get a certificate if you finish the course with a \\"live\\" cohort.\\n\\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\\n\\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n   

In [13]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [15]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf your goal is just to learn, you can start anytime: the videos, materials, and homework are available.\n\nIf you want a certificate, you’ll need to join while the course is running and submit your project before submissions close, because peer review is only available during the live cohort.'

In [16]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(774, 71)

In [17]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [18]:
response.output

[ResponseOutputMessage(id='msg_0bbea0dc257b9535006a35dd845f7c819bb270b3b08f367044', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.\n\nIf your goal is just to learn, you can start anytime: the videos, materials, and homework are available.\n\nIf you want a certificate, you’ll need to join while the course is running and submit your project before submissions close, because peer review is only available during the live cohort.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [19]:
# Agentic Loop
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [20]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [21]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment late registration"}
function_call: search {"query":"course FAQ enrollment join course new student register join discovered course"}


In [22]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

A couple of important notes:
- You can start learning even if you just discovered it.
- If you want a certificate, you need to submit your project while submissions are still open.
- If you’re following it in self-paced mode, you can learn the material, but certificates are only available for the live cohort.

If you want, I can also help you figure out the best way to start catching up. Is there anything else you’d like to explore?


In [24]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [25]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Ollama run locally install local model course FAQ"}
function_call: search {"query":"Olama locally run local model FAQ"}
function_call: search {"query":"Ollama local setup macOS Windows Linux FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. Install Ollama:
   - macOS: download the `.pkg` from https://ollama.com/download
   - Windows: download the `.msi` from https://ollama.com/download
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Start a model:
   ```bash
   ollama run llama3
   ```
   This downloads the model and starts a local chat session.

3. Check that the local server is running:
   ```bash
   curl http://localhost:11434
   ```
   You should get a response showing available models.

4. If you want to use it from Python:
   ```bash
   pip install ollama
   ```
   Then:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user"

'To run Ollama locally:\n\n1. Install Ollama:\n   - macOS: download the `.pkg` from https://ollama.com/download\n   - Windows: download the `.msi` from https://ollama.com/download\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Start a model:\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model and starts a local chat session.\n\n3. Check that the local server is running:\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response showing available models.\n\n4. If you want to use it from Python:\n   ```bash\n   pip install ollama\n   ```\n   Then:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a “connection refused” error later, restart the server with:\n```bash\nollama serve\n```\nor in a notebook:\n```bash\n!nohup ollama serve

In [26]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course can I join discovered the course enrollment late join FAQ"}
iteration #2...
function_call: search {"query":"LLM Zoomcamp certificate project submission while accepting submissions join course late self-paced certificate peer-review"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open. Also, certificates are only available if you complete the course with the live cohort, not in self-paced mode.

If you'd like, I can also help you find the course materials, deadlines, or the best way to get started. Is there anything else you want to explore?


"Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open. Also, certificates are only available if you complete the course with the live cohort, not in self-paced mode.\n\nIf you'd like, I can also help you find the course materials, deadlines, or the best way to get started. Is there anything else you want to explore?"

In [27]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess queen's gambit opening definition"}
function_call: search {"query":"queen gambit course FAQ queen's gambit"}
function_call: search {"query":"what is queen gambit"}
iteration #2...
ASSISTANT:
A **Queen’s Gambit** is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers the c-pawn to try to distract Black’s d-pawn and gain better control of the center.

A few quick points:
- It’s called a **gambit** because White may be offering material.
- It’s one of the most famous and solid chess openings.
- Black can accept the pawn or decline it.

If you meant the **TV show/book** *The Queen’s Gambit*, I can explain that too. Want a simple breakdown of the opening moves or the show?


'A **Queen’s Gambit** is a chess opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the c-pawn to try to distract Black’s d-pawn and gain better control of the center.\n\nA few quick points:\n- It’s called a **gambit** because White may be offering material.\n- It’s one of the most famous and solid chess openings.\n- Black can accept the pawn or decline it.\n\nIf you meant the **TV show/book** *The Queen’s Gambit*, I can explain that too. Want a simple breakdown of the opening moves or the show?'

In [28]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"gambit chess opening queen gambit"}
iteration #3...
function_call: search {"query":"queen's gambit chess course FAQ"}
iteration #4...
ASSISTANT:
I couldn’t find a course FAQ entry for “queen gambit,” and it doesn’t look like a course-related topic.

If you meant **“queen’s gambit”** in chess, that’s outside the course FAQ I can use here. If you want, I can help you look for a course-related question instead. Are there other areas you want to explore?


'I couldn’t find a course FAQ entry for “queen gambit,” and it doesn’t look like a course-related topic.\n\nIf you meant **“queen’s gambit”** in chess, that’s outside the course FAQ I can use here. If you want, I can help you look for a course-related question instead. Are there other areas you want to explore?'

In [29]:
!uv add toyaikit

⠹                                                                               Resolved 127 packages in 2.17s
⠙ Preparing packages... (0/7)                                                   ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/7)-------------------     0 B/21.96 KiB           
⠙ Preparing packages... (0/7)---------- 14.80 KiB/21.96 KiB         
⠙ Preparing packages... (0/7)---------- 14.80 KiB/21.96 KiB         
docstring-parser     ------------------------------ 14.80 KiB/21.96 KiB
⠙ Preparing packages... (0/7)-------------------     0 B/73.18 KiB           
docstring-parser     ------------------------------ 14.80 KiB/21.96 KiB
⠙ Preparing packages... (0/7)------------------- 14.84 KiB/73.18 KiB         
docstring-parser     ------------------------------ 14.80 KiB/21.96 KiB
⠙ Preparing packages... (0/7)------------------- 14.84 KiB/73.18 KiB         
docstring-parser     ------------------------------ 14.80 KiB/

In [30]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [31]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [32]:
# we dont really need tool schema to be defined manually. We can write tool (defining data types) like below:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [33]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [34]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [35]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [36]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [37]:
result.cost

CostInfo(input_cost=Decimal('0.002655'), output_cost=Decimal('0.0013095'), total_cost=Decimal('0.0039645'))

In [38]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama run locally local install setup course FAQ"}', call_id='

In [39]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [40]:
runner.run()

-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='whats the meaning of life?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"meaning of life"}', call_id='call_b8Fxt2Q